In [18]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [19]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)


(array([0, 1]), array([900, 100]))

In [20]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

In [21]:

from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [22]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'lbfgs'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [23]:

reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [24]:

import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [25]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

# Initialize MLflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# Create experiment
mlflow.set_experiment("deployment model")

# Trusted type for your own Decision Tree-based models
trusted_types = ["sklearn.tree._tree.Tree"]

for i, element in enumerate(models):

    model_name = element[0]
    params = element[1]
    model = element[2]

    report = reports[i]

    with mlflow.start_run(run_name=model_name):

        # Log parameters
        mlflow.log_params(params)

        # Log metrics
        mlflow.log_metrics({
            "accuracy": float(report["accuracy"]),
            "recall_class_1": float(report["1"]["recall"]),
            "recall_class_0": float(report["0"]["recall"]),
            "f1_score_macro": float(report["macro avg"]["f1-score"])
        })

        # Log models
        if "XGB" in model_name:

            mlflow.xgboost.log_model(
                model,
                artifact_path="model"
            )

        else:

            mlflow.sklearn.log_model(
                model,
                artifact_path="model",
                skops_trusted_types=trusted_types
            )

    print(f"{model_name} logged successfully!")

print("All models logged successfully!")

2026/10/30 09:53:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/30 09:54:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/3/runs/9fed8a0f459849feaf6141cf91088148
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Logistic Regression logged successfully!


2026/10/30 09:54:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/3/runs/bfe1d50d55c24272abbc2e5123f7f207
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
Random Forest logged successfully!


2026/10/30 09:54:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/3/runs/2dfc40a0c55c414f92d0d5c2a2e94655
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
XGBClassifier logged successfully!
🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/3/runs/039128cc83a0467f8997feee65abf538
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
XGBClassifier With SMOTE logged successfully!
All models logged successfully!


In [26]:
model_name = 'XGB-Smote'
run_id=input('Please type RunID')
model_uri = f'runs:/{run_id}/model'

with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/10/30 09:55:15 WARNING mlflow.tracking._model_registry.fluent: Run with id 039128cc83a0467f8997feee65abf538 has no artifacts at artifact path 'model', registering model based on models:/m-b86a534e357d4eed89904b0daf80cae8 instead
2026/10/30 09:55:15 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 3


🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/3/runs/039128cc83a0467f8997feee65abf538
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


Created version '3' of model 'XGB-Smote'.


In [27]:
model_name = 'XGBClassifier'
run_id=input('Please type RunID')
model_uri = f'runs:/{run_id}/model'

with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'XGBClassifier' already exists. Creating a new version of this model...
2026/10/30 09:57:27 WARNING mlflow.tracking._model_registry.fluent: Run with id 204a7e46a0e343c0a25ce9f076e8f6b1 has no artifacts at artifact path 'model', registering model based on models:/m-308283a57d8d462889de78cd732e3399 instead
2026/10/30 09:57:27 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGBClassifier, version 2


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/3/runs/204a7e46a0e343c0a25ce9f076e8f6b1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


Created version '2' of model 'XGBClassifier'.


In [33]:
model_name = 'XGB-Smote'
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [37]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

client = mlflow.MlflowClient()

for m in client.search_registered_models():
    print("Model Name:", m.name)

    for v in client.search_model_versions(f"name='{m.name}'"):
        print("Version:", v.version)
        print("Aliases:", v.aliases)

Model Name: XGB-Smote
Version: 3
Aliases: []
Version: 2
Aliases: []
Version: 1
Aliases: []
Model Name: XGBClassifier
Version: 2
Aliases: []
Version: 1
Aliases: []


In [38]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

client = mlflow.MlflowClient()

model_name = "XGB-Smote"

client.set_registered_model_alias(
    name=model_name,
    alias="appserver",
    version="3"
)

print("Alias created successfully!")

Alias created successfully!


In [39]:
current_model_uri = f"models:/{model_name}@appserver"

production_model_name = "finalproduction"

copied_model = client.copy_model_version(
    src_model_uri=current_model_uri,
    dst_name=production_model_name
)

print("Model copied successfully!")
print("Production Model:", copied_model.name)
print("Production Version:", copied_model.version)

Model copied successfully!
Production Model: finalproduction
Production Version: 1


Successfully registered model 'finalproduction'.
Copied version '3' of model 'XGB-Smote' to version '1' of model 'finalproduction'.


In [40]:
for model in client.search_model_versions(
    "name='finalproduction'"
):
    print("Model:", model.name)
    print("Version:", model.version)
    print("Status:", model.status)

Model: finalproduction
Version: 1
Status: READY


In [41]:
import mlflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")

client = mlflow.MlflowClient()

production_model_name = "finalproduction"

# Get registered model versions
versions = client.search_model_versions(
    "name='finalproduction'"
)

latest_version = max(
    versions,
    key=lambda x: int(x.version)
)

# Assign production alias
client.set_registered_model_alias(
    name=production_model_name,
    alias="production",
    version=latest_version.version
)

print("Production alias assigned successfully!")

Production alias assigned successfully!
